In [0]:
from pyspark.sql import functions as F

df_silver = spark.table("internet_fijo_elt.silver.conexiones_internet_fijo")

display(df_silver.limit(10))


In [0]:
# Traer las dimensiones para hacer join y obtener los IDs
fact_base = (
    df_silver
    .groupBy(
        "periodo",
        "empresa",
        "departamento",
        "provincia",
        "distrito",
        "tecnologia",
        "segmento"
    )
    .agg(
        F.sum("conexiones").alias("conexiones")
    )
)

In [0]:
# Cargar dimensiones
dim_periodo = spark.table("internet_fijo_elt.gold.dim_periodo")
dim_empresa = spark.table("internet_fijo_elt.gold.dim_empresa")
dim_tecnologia = spark.table("internet_fijo_elt.gold.dim_tecnologia")
dim_segmento = spark.table("internet_fijo_elt.gold.dim_segmento")
dim_ubicacion = spark.table("internet_fijo_elt.gold.dim_ubicacion")

fact_conexiones = (
    fact_base
    .join(dim_periodo, fact_base["periodo"] == dim_periodo["periodo"], "left")
    .join(dim_empresa, fact_base["empresa"] == dim_empresa["nombre_empresa"], "left")
    .join(dim_tecnologia, fact_base["tecnologia"] == dim_tecnologia["tecnologia"], "left")
    .join(dim_segmento, fact_base["segmento"] == dim_segmento["segmento"], "left")
    .join(
        dim_ubicacion,
        (fact_base["departamento"] == dim_ubicacion["departamento"]) &
        (fact_base["provincia"] == dim_ubicacion["provincia"]) &
        (fact_base["distrito"] == dim_ubicacion["distrito"]),
        "left"
    )
)

In [0]:
fact_conexiones = fact_conexiones.select(
    "fecha_id",
    "id_empresa",
    "id_ubicacion",
    "id_tecnologia",
    "id_segmento",
    "conexiones"
)

In [0]:
fact_conexiones.write.format("delta").mode("overwrite").saveAsTable(
    "internet_fijo_elt.gold.fact_conexiones"
)

In [0]:
print("Registros después de agregar:", fact_base.count())

In [0]:
fact_conexiones = (
    fact_base
    .join(dim_periodo, fact_base["periodo"] == dim_periodo["periodo"], "left")
    .join(dim_empresa, fact_base["empresa"] == dim_empresa["nombre_empresa"], "left")
    .join(dim_tecnologia, fact_base["tecnologia"] == dim_tecnologia["tecnologia"], "left")
    .join(dim_segmento, fact_base["segmento"] == dim_segmento["segmento"], "left")
    .join(
        dim_ubicacion,
        (fact_base["departamento"] == dim_ubicacion["departamento"]) &
        (fact_base["provincia"] == dim_ubicacion["provincia"]) &
        (fact_base["distrito"] == dim_ubicacion["distrito"]),
        "left"
    )
)

In [0]:
fact_conexiones = fact_conexiones.select(
    "fecha_id",
    "id_empresa",
    "id_ubicacion",
    "id_tecnologia",
    "id_segmento",
    "conexiones"
)

In [0]:
print("Registros Fact:", fact_conexiones.count())

In [0]:
display(fact_conexiones.limit(10))

In [0]:
fact_conexiones.filter(
    F.col("fecha_id").isNull() |
    F.col("id_empresa").isNull() |
    F.col("id_ubicacion").isNull() |
    F.col("id_tecnologia").isNull() |
    F.col("id_segmento").isNull()
).count()

In [0]:
fact_conexiones.write.format("delta").mode("overwrite").saveAsTable(
    "internet_fijo_elt.gold.fact_conexiones"
)

In [0]:
print("Fact:", spark.table("internet_fijo_elt.gold.fact_conexiones").count())

In [0]:
gold_total = (
    spark.table("internet_fijo_elt.gold.fact_conexiones")
    .agg(F.sum("conexiones").alias("total"))
    .collect()[0]["total"]
)

print("Total conexiones Gold:", gold_total)